# Fine-Tuning and Training Kite 1.0 with a Qwen Backbone in Google Colab

Welcome! This notebook provides a complete pipeline to assemble, train, and test **Kite 1.0** using a pre-trained **Qwen2.5-0.5B-Instruct** backbone. 

By replacing the massive DeepSeek-V3 671B backbone with Qwen-0.5B:
- The model fits completely in the **15GB VRAM of a standard Colab T4 GPU**.
- The model uses Qwen's real pre-trained weights and vocabulary, allowing it to output **actual English text** rather than placeholder dummy tokens.

## Step 1: Clone the GitHub Repository
Clone your repository to the Colab machine to load the source files.

In [ ]:
# Your GitHub repository URL
REPO_URL = "https://github.com/paswans05/kite1.0.git"

import os
if not os.path.exists("train_kite.py"):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} kite_repo
    %cd kite_repo
    print("Switched working directory to:", os.getcwd())
else:
    print("Already inside the repository.")

In [ ]:
# If you already cloned, run this to get the latest updates and clean cache
!git pull
!rm -rf /root/.cache/huggingface/modules/transformers_modules/

## Step 2: Install Dependencies
Install the required libraries for model setup, video/image processing, and inference.

In [ ]:
!pip install -q transformers accelerate peft decord pillow einops pydantic tiktoken

## Step 3: Assemble Kite with Pre-trained Qwen Backbone
Run the assembly script to load the pre-trained weights of `Qwen/Qwen2.5-0.5B-Instruct` from Hugging Face and merge it with the Kite vision tower and projector configurations.

In [ ]:
# Load pre-trained Qwen weights and save the assembled model base
!python create_kite_qwen.py \
    --lm_model_id "Qwen/Qwen2.5-0.5B-Instruct" \
    --output_dir "./kite_qwen_base"

## Step 4: Fine-Tune the Multimodal Projector (Training)
Run `train_kite.py` in `projector_only` mode pointing to `./kite_qwen_base`. This freezes the pre-trained text backbone and trains only the Multimodal Projector and Vision Tower on the GPU.

You can choose to train with a dummy dataset (for verification) or with your own actual dataset JSON file.

In [ ]:
# Option A: Train with the dummy dataset (quick run to verify pipeline)
!python train_kite.py \
    --model_path "./kite_qwen_base" \
    --mode "projector_only" \
    --dummy \
    --epochs 1 \
    --lr 2e-5 \
    --batch_size 1 \
    --grad_accum 4 \
    --output_dir "./kite_qwen_trained"

In [ ]:
# Option B: Train with actual local JSON dataset (change data_path to your uploaded dataset JSON)
!python train_kite.py \
    --model_path "./kite_qwen_base" \
    --mode "projector_only" \
    --data_path "./sample_data.json" \
    --epochs 3 \
    --lr 2e-5 \
    --batch_size 2 \
    --grad_accum 4 \
    --output_dir "./kite_qwen_trained"

In [ ]:
# Option C: Train loading a dataset directly from Hugging Face Hub (change the data_path to your dataset ID)
!pip install -q datasets
!python train_kite.py \
    --model_path "./kite_qwen_base" \
    --mode "projector_only" \
    --data_path "liuhaotian/LLaVA-Instruct-150K" \
    --epochs 3 \
    --lr 2e-5 \
    --batch_size 2 \
    --grad_accum 4 \
    --output_dir "./kite_qwen_trained"

## Step 5: Test Model with Real English Responses (Inference)
Run the testing script using an online image URL. The model will load the fine-tuned checkpoint, download the image, and output **actual English text**!

In [ ]:
# Test the model using a public image URL
!python test_kite.py \
    --model_path "./kite_qwen_trained" \
    --image_path "https://i.ibb.co/sdf0DN54/Nitro-Wallpaper-01-3840x2400.jpg" \
    --prompt "Describe what is in this image."

## Step 6: Upload the Fine-Tuned Model to Hugging Face Hub
Upload the folder of your fine-tuned model (`./kite_qwen_trained`) directly to your Hugging Face account with the model name `kite-i0.5b`.

In [ ]:
# Replace "hf_YOUR_WRITE_TOKEN_HERE" with your actual Hugging Face write token
!python upload_to_hf.py \
    --folder_path "./kite_qwen_trained" \
    --repo_id "paswans05/kite-i0.5b" \
    --token "hf_YOUR_WRITE_TOKEN_HERE"